# WP0 · Data

**If you work on the model or on evaluation, Part A is all you need.** Run the setup cell and Part A: they
produce the aligned corpora and the 16 subword vocabularies (only what is missing), check your copy against
the team's, and A3 says which file or function to use for which experiment.

Part B is WP0's own analysis, for the report: what a jointly learned vocabulary costs Azerbaijani (`az`) and
Belarusian (`be`), and whether that cost can be recovered. Nothing in Part A depends on it.

In [ ]:
import csv, json, shutil, subprocess, sys
from pathlib import Path

import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter, PercentFormatter
from IPython.display import Markdown, display

ROOT = Path.cwd()
while not (ROOT / "src" / "preprocessing" / "align.py").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
PREP, RESULTS = ROOT / "src" / "preprocessing", ROOT / "results" / "metrics" / "preprocessing"
PROC, VOCABS = ROOT / "data" / "processed", ROOT / "artifacts" / "vocabularies"
sys.path.insert(0, str(PREP))
import align

SPLITS, LANGS6 = ["train", "dev", "test"], align.PROJECT_LANGS
PY = ["uv", "run", "python"] if shutil.which("uv") else [sys.executable]       # the Makefile's choice

# Figures: one hue for magnitude, two categorical hues (Unigram / BPE), a one-hue ramp for ordered parts.
INK, INK2, MUTED, GRID, AXIS, SURFACE = "#0b0b0b", "#52514e", "#898781", "#e1e0d9", "#c3c2b7", "#fcfcfb"
BLUE, ORANGE = "#2a78d6", "#eb6834"                                              # Unigram, BPE
RAMP = {"affix": "#256abf", "other alphabetic": "#5598e7", "generic": "#86b6ef"}  # most → least meaningful sharing
plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE, "figure.dpi": 110,
    "font.family": "sans-serif", "font.size": 10, "text.color": INK,
    "axes.edgecolor": AXIS, "axes.linewidth": 0.8, "axes.labelcolor": INK2,
    "axes.titlecolor": INK, "axes.titlesize": 10.5, "axes.titleweight": "bold", "axes.titlelocation": "left",
    "axes.spines.top": False, "axes.spines.right": False,
    "xtick.color": AXIS, "ytick.color": AXIS, "xtick.labelcolor": INK2, "ytick.labelcolor": INK2,
    "grid.color": GRID, "grid.linewidth": 0.8, "grid.linestyle": "-", "legend.frameon": False,
})

def tidy(ax, xgrid=False):
    ax.tick_params(length=0)
    if xgrid:
        ax.grid(axis="x", zorder=0); ax.set_axisbelow(True)

def heading(fig, text):
    fig.suptitle(text, x=0.01, ha="left", fontsize=12, fontweight="bold", color=INK)

## A · The data, ready to use

**A1** runs a pipeline step only if one of its outputs is missing, or if a step it reads from was just
regenerated. The first run downloads `TED.tar` (673 MB) and trains the vocabularies; after that it takes
seconds. **A2** checks the result against the team's. Two rules come with the data:

- **A blank line is a placeholder, not dirty data.** It means "no translation at this position". Drop it
  and every later position shifts by one, and nothing errors. So parallel text is read only by *position*:
  through `align.py` (`positions` / `pairs` / `triples`) or `src/data.py`'s `get_language_pairs`, which
  delegates to it. **`data/processed/vocab_training/` has its blank lines removed and feeds vocabulary
  training only.**
- **Only `manifest.json` and `index.json` are committed.** The corpus is CC-BY-NC-ND-4.0, so everyone
  regenerates it locally and A2 compares the hashes with the committed files (not the local copies, which
  `emit` and `train_vocabs.py` rewrite whenever they run). Both checks must say `OK` before any experiment
  you want to compare with someone else's.

In [ ]:
manifest = json.loads((PROC / "parallel" / "manifest.json").read_text(encoding="utf-8"))
index = json.loads((VOCABS / "index.json").read_text(encoding="utf-8"))
vocabs = lambda model: [VOCABS / t / "tokenizer.json" for t, e in index.items() if e["model"] == model]

# step: (script and flags, the files it must leave behind, the steps whose output it reads)
DATA_STEPS = {
    "extract":        (["align.py", "extract"], [align.RAW / s / f"{l}.txt" for s in SPLITS for l in LANGS6], []),
    "prep_data":      (["prep_data.py"], [PROC / "vocab_training" / f"{l}.{s}.txt" for s in SPLITS for l in LANGS6], []),
    "emit":           (["align.py", "emit"], [PROC / "parallel" / f for f in manifest["files"]], ["extract"]),
    "pivot":          (["align.py", "pivot"], [PROC / "pivot" / f"{'-'.join(p)}.{s}.tsv" for p in align.PIVOTS for s in SPLITS], ["extract"]),
    "vocabs unigram": (["train_vocabs.py", "--model", "unigram"], vocabs("unigram"), ["prep_data"]),
    "vocabs bpe":     (["train_vocabs.py", "--model", "bpe"], vocabs("bpe"), ["prep_data"]),
}
ANALYSIS_STEPS = {                                                               # run in B2; A does not need them
    "cost":           (["cost_analysis.py"], [RESULTS / "cost_metrics.csv", RESULTS / "cost_report.md"], ["vocabs unigram", "vocabs bpe"]),
    "directionality": (["directionality.py"], [RESULTS / "directionality.csv", RESULTS / "directionality_report.md"], ["cost"]),
    "sensitivity":    (["sensitivity.py"], [RESULTS / "sensitivity.md"], ["cost"]),
}

def run(*cmd):
    with subprocess.Popen(cmd, cwd=ROOT, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True) as p:
        for line in p.stdout:
            print("      " + line, end="")
    if p.returncode:
        raise RuntimeError(f"{' '.join(map(str, cmd))} exited with {p.returncode}")

ran = set()
def produce(steps):
    for name, (cmd, outputs, reads) in steps.items():
        missing = sum(not p.exists() for p in outputs)
        if not missing and not ran & set(reads):
            print(f"ok    {name:15s} {len(outputs)} files present")
            continue
        print(f"run   {name:15s} " + (f"{missing} of {len(outputs)} files missing" if missing else "an input was just regenerated"))
        if name in ("extract", "prep_data") and not align.TAR.exists():
            run("make", "data/TED.tar")                                            # 673 MB, once
        run(*PY, PREP / cmd[0], *cmd[1:])
        ran.add(name)

produce(DATA_STEPS)

In [ ]:
# The alignment contract: in every split, all six languages have the same number of raw lines.
# `_raw` is align.py's own line reader, used directly because the line counts *are* the contract.
raw = {(s, l): align._raw(s, l) for s in SPLITS for l in LANGS6}                    # B1 reuses it
for s in SPLITS:
    sizes = {l: len(raw[(s, l)]) for l in LANGS6}
    assert len(set(sizes.values())) == 1, f"alignment broken in {s}: {sizes}"
print("OK    every split has the same number of lines in all six languages: "
      + ", ".join(f"{s} {len(raw[(s, 'en')]):,}" for s in SPLITS))

# Same data and same segmentation as the team: hashes against the committed manifest and index.
import tempfile, verify
from contextlib import redirect_stdout
from io import StringIO

def committed(rel):
    """The file as committed in git; the working copy if this is not a git checkout."""
    try:
        r = subprocess.run(["git", "show", f"HEAD:{rel}"], cwd=ROOT, capture_output=True)
    except OSError:
        return ROOT / rel
    if r.returncode:
        return ROOT / rel
    p = Path(tempfile.mkdtemp()) / Path(rel).name
    p.write_bytes(r.stdout)
    return p

verdict = lambda text: print(*[l for l in text.splitlines() if l.startswith(("OK", "FAIL", "->"))], sep="\n")
verdict(subprocess.run([*PY, PREP / "verify.py", "--manifest", committed("data/processed/parallel/manifest.json"),
                        "--dir", PROC / "parallel"], cwd=ROOT, capture_output=True, text=True).stdout)
with redirect_stdout(StringIO()) as out:
    verify.verify_vocabs(committed("artifacts/vocabularies/index.json"), VOCABS, quiet=True)
verdict(out.getvalue())

### A3 · What to use for what

Training and evaluation code gets everything through one object, `TedData` in `src/data.py`. It fetches and
unpacks the corpus if it is missing (on Colab too), then reads through `align.py`, so what it returns is what
A2 just verified.

| You work on | Use | Notes |
|---|---|---|
| **NLLB fine-tuning** | `ted.examples(split, DIRECTIONS, CODES)` | returns exactly the rows `baselines_nllb.ipynb` builds for its JSONL today (tested line for line), so its numbers stand. For the helper languages, add `"tr-en"`, `"en-tr"` … to `DIRECTIONS`: no new code |
| **a model trained from scratch** | `ted.pairs(split, src, tgt)`, then `ted.segment(texts, lang, vocab)` | `vocab` is one of the 16 directories in `artifacts/vocabularies/`: `V3_8000_unigram` is the main experiment, B0 says what each one isolates, and `V1_az` / `V1_be` cover their own language only. Do not pair up `tokenize_corpus.py` output: it segments `vocab_training/`, whose files do not line up |
| **pivot / helper experiments** | `align.triples(split, ["az", "tr", "en"])`, or `data/processed/pivot/az-tr-en.{split}.tsv` | also `be-uk-en` and `be-ru-en` |
| **evaluation** | `ted.references(direction)`, and `ted.check_references(folder)` for a system's output | every `<src>-<tgt>.ref` must hold the team's test references; the last cell of Part A checks every system |

Every pair carries its position number (the first element of each tuple, and the `.pos` file), which joins
it to the other pairs and to the triples.

In [ ]:
# Every delivered pair: sentence pairs per split (line counts from the manifest).
print(f"{'pair':8s}" + "".join(f"{s:>9s}" for s in SPLITS))
for a, b in align.DELIVER_PAIRS:
    print(f"{a}-{b:5s}" + "".join(f"{manifest['files'][f'{a}-{b}.{s}.{a}']['lines']:>9,}" for s in SPLITS))

# What training and evaluation code writes.
sys.path.insert(0, str(ROOT))
from src.data import TedData

ted = TedData()                                                                   # fetches / unpacks only what is missing
pairs = ted.pairs("test", "az", "en")                                             # [(position, az, en), ...]
pos, az, en = pairs[0]
print(f"\n{len(pairs):,} az-en test pairs. The first, at position {pos}:")
print("  az      :", az)
print("  en      :", en)
print("  az · V3 :", " ".join(ted.segment([az], "az")[0]))
print("  en · V3 :", " ".join(ted.segment([en], "en")[0]))
print("  as an NLLB example:", ted.examples("test", ["az-en"], {"az": "azj_Latn", "en": "eng_Latn"})[0])

In [ ]:
# Evaluation: is every system scored against the same references? One line per results/translations/<system>/<src>-<tgt>.ref
TRANS = ROOT / "results" / "translations"
systems = sorted(d for d in TRANS.iterdir() if d.is_dir()) if TRANS.exists() else []
if not systems:
    print(f"no system output under {TRANS.relative_to(ROOT)} yet")
for system in systems:
    for name, verdict in ted.check_references(system).items():
        print(f"{system.name + '/' + name:32s} {verdict}")

## B · Analysis (WP0 and the report)

**Question:** what does a *jointly learned* subword vocabulary cost the low-resource languages
(Azerbaijani `az`, Belarusian `be`), and can that cost be recovered? WP0 answers at the level of the
vocabulary, with CPU-only string statistics. Whether a lower cost becomes better translation is the model
side's BLEU / chrF, not WP0. B1 is the corpus, B2 the results, B3 which of those numbers survive a change
of setting.

### B0 · Definitions

#### How the data is built

```
TED.tar  (davidstap/ted_talks, 6 languages: en az be tr uk ru)
   │
   ├─ align.py extract ──▶ data/raw/{split}/{lang}.txt                   blank lines KEPT: the alignment contract
   │     ├─ align.py emit  ──▶ data/processed/parallel/{pair}.{split}.*   the parallel corpora (+ manifest.json)
   │     └─ align.py pivot ──▶ data/processed/pivot/*.tsv                 az-tr-en / be-uk-en / be-ru-en triples
   │
   └─ prep_data.py ──▶ data/processed/vocab_training/{lang}.{split}.txt  blank lines DROPPED: vocabulary training only
         └─ train_vocabs.py ──▶ artifacts/vocabularies/<tag>/tokenizer.json (+ index.json)
               ├─ cost_analysis.py  ──▶ results/metrics/preprocessing/cost_metrics.csv, cost_report.md   C1–C6
               ├─ directionality.py ──▶ …/directionality.csv, directionality_report.md   translation evidence, C3 split
               └─ sensitivity.py    ──▶ …/sensitivity.md                                 which numbers move with a setting

verify.py ──▶ your parallel corpora vs manifest.json, your vocabularies vs index.json
```

The scripts live in `src/preprocessing/`. `make all` runs the same steps from the shell (all but
`sensitivity`), but re-runs every one of them unconditionally.

#### The vocabularies V1–V5: one thing varies

Vocabulary size (8,000), segmentation model, special tokens and seed are fixed; **only the training
languages change**, so any difference between two variants is attributable to sharing itself. Each variant
is trained as Unigram and as BPE, and a number is only compared with a zero-point of the same model.

| # | Trained on | Role |
|---|---|---|
| **V1** | az only / be only | the **zero-point** every cost is measured from |
| **V2** | az + be + en | shares English only; no helper language seen |
| **V3** | az + be + **tr + uk + ru** + en | **the main experiment**: learn it on all of them at once |
| **V4** | as V3, az / be upsampled with α ∈ {0.2, 0.5, 0.8} | **recovery R1** (α = 1 is V3). Works if C1 returns toward V1 without `morph` collapsing |
| **V5** | as V3, after a character map az→tr, be/uk→ru | **recovery R2**, "make the strings identical". Works if sharing rises on morphology, not on single letters |

The helpers are fixed, `HELPER = {az: [tr], be: [uk, ru]}`, and az and be are compared per helper, never in
aggregate. Per model that is V1 twice, V4 three times: 16 vocabularies in all.

#### The metrics C1–C6

Per (vocabulary, language), on the test split, for az and be only.

| # | Metric | Definition | Reads as |
|---|---|---|---|
| **C1** | fertility | subword pieces per whitespace word | the headline: how fragmented the language becomes |
| **C2** | inflation | token count of the same sentences ÷ the language's V1 count, same model | the same content costs a longer sequence |
| **C3** | shared-piece share | fraction of the language's tokens landing on a piece the helper also uses | the benefit side, per helper. C3 = `morph` (alphabetic pieces; `affix` is the productive-suffix part of it) + `generic` (single letters, digits, punctuation) |
| **C4** | fragmentation tail | fraction of words cut into ≥ 4 pieces | the agglutinative worst case |
| **C5** | allocation share | fraction of the 8,000 slots whose most frequent user is this language | the mechanism: how much of the vocabulary the language gets |
| **C6** | shared-piece productivity | of the shared tokens: `useful` = productive suffixes, `fragment` = single letters, `junk` = no letters | whether the sharing is worth anything |

### B1 · The corpus

In [ ]:
fill = {k: sum(1 for x in v if x.strip()) / len(v) for k, v in raw.items()}
order = sorted(LANGS6, key=lambda l: fill[("train", l)])                        # largest at the top

fig, axes = plt.subplots(1, 3, figsize=(10, 3.1), sharey=True)
for ax, s in zip(axes, SPLITS):
    vals = [fill[(s, l)] for l in order]
    bars = ax.barh(order, vals, height=0.55, color=BLUE, zorder=3)
    for b, v in zip(bars, vals):
        ax.text(v + 0.02, b.get_y() + b.get_height() / 2, f"{v:.1%}", va="center", fontsize=9, color=INK2)
    ax.set_xlim(0, 1.25); ax.set_xticks([0, 0.5, 1]); ax.xaxis.set_major_formatter(PercentFormatter(1.0, decimals=0))
    ax.set_title(f"{s} · {len(raw[(s, 'en')]):,} aligned positions"); tidy(ax, xgrid=True)
heading(fig, "Share of aligned positions that carry text, per language")
fig.tight_layout(rect=(0, 0, 1, 0.93)); plt.show()

In [ ]:
groups = [("main line", [("az", "en"), ("be", "en")]),
          ("helper training", [("tr", "en"), ("uk", "en"), ("ru", "en")]),
          ("RQ1 direct", [("be", "uk"), ("be", "ru")]),
          ("auxiliary", [("az", "tr"), ("az", "be")])]
labels, counts, ys, y = [], [], [], 0.0
for name, prs in groups:
    for a, b in prs:
        labels.append(f"{name} · {a}-{b}"); counts.append(len(align.positions("train", [a, b]))); ys.append(y); y -= 1
    y -= 0.5
fig, ax = plt.subplots(figsize=(8.5, 4))
ax.set_xscale("log"); ax.set_xlim(150, 1.2e6)
ax.hlines(ys, 150, counts, color=GRID, linewidth=1, zorder=2)
ax.scatter(counts, ys, s=70, color=BLUE, edgecolor=SURFACE, linewidth=2, zorder=3)
for lab, c, yy in zip(labels, counts, ys):
    ax.text(c * 1.25, yy, f"{c:,}", va="center", fontsize=9, color=INK2)
ax.set_yticks(ys); ax.set_yticklabels(labels)
ax.set_xticks([1e3, 1e4, 1e5, 1e6]); ax.xaxis.set_major_formatter(FuncFormatter(lambda v, _: f"{v:,.0f}"))
ax.set_xlabel("training sentence pairs (log scale)"); tidy(ax, xgrid=True)
heading(fig, f"Training pairs per language pair: a {max(counts) / min(counts):,.0f}× range under one name, \"TED\"")
fig.tight_layout(rect=(0, 0, 1, 0.93)); plt.show()

In [ ]:
# Stage directions such as "(Applause)": where English has one, does the same position have one too?
# Walks the whole training split, so give it a minute.
import directionality

rows = directionality.inheritance_table("train", ["az", "be", "tr", "uk", "ru"])[::-1]
fig, ax = plt.subplots(figsize=(7, 2.8))
bars = ax.barh([r["lang"] for r in rows], [r["stage_direction_inheritance"] for r in rows], height=0.55, color=BLUE, zorder=3)
for b, r in zip(bars, rows):
    ax.text(r["stage_direction_inheritance"] + 0.015, b.get_y() + b.get_height() / 2,
            f"{r['stage_direction_inheritance']:.1%}   ({r['both_have_stage_direction']:,} of {r['en_with_stage_direction']:,})",
            va="center", fontsize=9, color=INK2)
ax.set_xlim(0, 1.5); ax.set_xticks([0, 0.5, 1]); ax.xaxis.set_major_formatter(PercentFormatter(1.0, decimals=0)); tidy(ax, xgrid=True)
heading(fig, "Where the English side has a stage direction, the same position has one too (train)")
fig.tight_layout(rect=(0, 0, 1, 0.9)); plt.show()

**Low-resource by measurement, not by label.** All six languages share the same aligned positions, and az
and be fill only a small fraction of them; tr / uk / ru fill tens of times more. That is
what makes the helpers useful *and* dangerous as competitors, and it is the mechanism C5 measures: one fixed
8,000-slot vocabulary, filled by raw frequency, so whoever has more text takes more of it. dev and test are
proportionally different from train, so do not read the test set as a miniature of the training set.

**Compare az and be per helper, never in aggregate.** be has two helpers (uk, ru), az one (tr), with
different volumes; that asymmetry is the natural experiment RQ1 rests on. The main line is small, and
`az-be` is pivot material only, not training data.

**Every non-English side is a translation of the English.** A native Azerbaijani corpus would not contain
`(Alqışlar)` at all; this one carries it wherever the English has `(Applause)`. Three consequences:

1. **The two directions are not symmetric tasks.** `X→en` recovers the source of a translation; `en→X`
   has to match one translator's choices, because that translation is the BLEU reference. A BLEU gap
   between directions is not by itself evidence of a data-volume gap.
2. **RQ1's "similarity" channel is confounded.** tr / uk / ru are translations of the same English
   sentences as az and be, so part of their shared structure comes from one shared translation process,
   not from the languages being related.
3. **C3's floor is inflated.** Same script and same English source mean shared letters, digits,
   punctuation and loanwords for free; B2 subtracts that.

### B2 · Results

The first cell computes the metrics if they are missing (`cost`, `directionality`, `sensitivity`: about
45 minutes the first time), then reads `cost_metrics.csv` and `directionality.csv` (test split, 8,000-piece
vocabularies). The printed lines
are the numbers each claim rests on, for both segmentation models, and every claim holds for both unless
the text says otherwise. The criterion the section applies:

> "The criterion for 'recoverable' is whether the shared pieces land exactly on az's high-frequency
> affixes. If C3 is high but C1 stays bad, then what is shared is irrelevant pieces and sharing is
> pure dilution — which is itself the more interesting finding."

In [ ]:
produce(ANALYSIS_STEPS)

R = {}
for f in ["cost_metrics.csv", "directionality.csv"]:                           # same (tag, lang) rows, different columns
    for r in csv.DictReader(open(RESULTS / f, encoding="utf-8")):
        R.setdefault((r["tag"], r["lang"]), {}).update(r)
HELPER, MODELS, LANGS = {"az": ["tr"], "be": ["uk", "ru"]}, ["unigram", "bpe"], ["az", "be"]

def x(v, m, lang, col):
    """One number. v is V1 V2 V3 V5, or V4a0.2 / V4a0.5 / V4a0.8."""
    t = f"V1_{lang}_8000_{m}" if v == "V1" else f"V4_8000_{m}_a{v[3:]}" if v.startswith("V4") else f"{v}_8000_{m}"
    return float(R[(t, lang)][col])

VARIANTS = [("V1 mono", "V1"), ("V2 +en", "V2"), ("V3 +helpers", "V3"), ("V4 α=0.8", "V4a0.8"),
            ("V4 α=0.5", "V4a0.5"), ("V4 α=0.2", "V4a0.2"), ("V5 translit", "V5")]
pct = FuncFormatter(lambda v, _: f"{v:.1%}" if v < 0.01 else f"{v:.0%}")
fig, axes = plt.subplots(2, 2, figsize=(10, 5.8), sharey=True)
for i, lang in enumerate(LANGS):
    for j, (col, title, log) in enumerate([("C1_fertility", "C1 · pieces per word (lower is better)", False),
                                           ("C5_alloc_share", "C5 · share of the 8,000 slots (log)", True)]):
        ax, ys = axes[i][j], list(range(len(VARIANTS)))[::-1]
        for m, colour in (("unigram", BLUE), ("bpe", ORANGE)):
            ax.plot([x(v, m, lang, col) for _, v in VARIANTS], ys, linewidth=0, marker="o", markersize=8, color=colour,
                    markeredgecolor=SURFACE, markeredgewidth=2, label=m, zorder=3)
        ax.set_yticks(ys); ax.set_yticklabels([n for n, _ in VARIANTS])
        if log: ax.set_xscale("log"); ax.set_xlim(4e-4, 1.0); ax.set_xticks([1e-3, 1e-2, 1e-1, 1.0]); ax.xaxis.set_major_formatter(pct)
        else: ax.set_xlim(1.4, 4.0)
        ax.set_title(f"{lang} · {title}"); tidy(ax, xgrid=True)
handles, names = axes[0][0].get_legend_handles_labels()
fig.legend(handles, names, loc="upper right", bbox_to_anchor=(0.99, 0.945), ncol=2, handletextpad=0.3)
heading(fig, "The cost (V1→V2→V3) and the two recovery arms, R1 (V4, α↓) and R2 (V5)")
fig.tight_layout(rect=(0, 0, 1, 0.94)); plt.show()

for m in MODELS:
    for lang in LANGS:
        c1, c4, c5 = ({v: x(v, m, lang, col) for v in ["V1", "V2", "V3"]} for col in ["C1_fertility", "C4_frag4_rate", "C5_alloc_share"])
        print(f"{m:8s}{lang}  C1 {c1['V1']:.2f}→{c1['V3']:.2f} (×{c1['V3'] / c1['V1']:.2f}), "
              f"{(c1['V2'] - c1['V1']) / (c1['V3'] - c1['V1']):.0%} of it already at V2   "
              f"C2 ×{x('V3', m, lang, 'C2_inflation_vs_baseline'):.2f}   C4 {c4['V1']:.0%}→{c4['V3']:.0%}   "
              f"C5 {c5['V1']:.1%}→{c5['V2']:.1%}→{c5['V3']:.2%}")

**The cost is large, and most of it is paid the moment you share at all.** Under V3 az and be need far more
pieces per word than under their own vocabulary, the same sentences cost a correspondingly longer sequence
(C2), and the share of words shattered into four or more pieces multiplies (C4); `cost_report.md` has the
word-level examples. The part that is easy to get wrong: most of that increase is already there under V2,
shared with English alone, and the three helper languages add little on top (under BPE be's fertility is
even slightly lower with them). The cost of a joint vocabulary is the cost of sharing an 8,000-slot budget
with a much larger corpus, whoever that corpus is.

**The mechanism is quota, not quality.** az's share of the slots collapses from V1 to V2 to V3 (right-hand
panels, log scale): joint training did not make az's pieces worse, it gave az almost none. C5 credits a slot
to whichever language uses it *most often*, so even the monolingual zero-point is well below 1: say "az's
frequency dominates", never "az owns". These are whole-corpus readings; B3 says why that must be stated.

*The V5 points in this figure and the next are not a valid measurement yet; see R2.*

In [ ]:
ROWS = [("V2 +en", "V2"), ("V3 +helpers", "V3"), ("V4 α=0.2", "V4a0.2"), ("V5 translit", "V5")]
fig, axes = plt.subplots(1, 2, figsize=(10.5, 3.2), sharey=True)
for ax, (lang, h) in zip(axes, [("az", "tr"), ("be", "uk")]):
    ys = list(range(len(ROWS)))[::-1]
    for y, (name, v) in zip(ys, ROWS):
        c = lambda part: x(v, "unigram", lang, f"C3_{part}_{h}")
        parts = {"affix": c("affix"), "other alphabetic": c("morph") - c("affix"), "generic": c("generic")}   # sums to C3
        left = 0.0
        for part, w in parts.items():
            ax.barh(y, w, left=left, height=0.55, color=RAMP[part], edgecolor=SURFACE, linewidth=2, zorder=3,
                    label=part if (y == ys[0] and lang == "az") else None)
            left += w
        ax.text(left + 0.015, y, f"C3 {c('total'):.2f} · generic {c('generic') / c('total'):.0%}", va="center", fontsize=9, color=INK2)
    ax.set_yticks(ys); ax.set_yticklabels([n for n, _ in ROWS]); ax.set_xlim(0, 1.32); ax.set_xticks([0, 0.25, 0.5, 0.75, 1.0])
    ax.set_title(f"{lang} vs {h} · Unigram"); tidy(ax, xgrid=True)
handles, names = axes[0].get_legend_handles_labels()
fig.legend(handles, names, loc="upper right", bbox_to_anchor=(0.99, 0.93), ncol=3, handlelength=1.2, columnspacing=1.0, handletextpad=0.4)
heading(fig, "What the shared tokens are: productive affixes, other alphabetic pieces, free script overlap")
fig.tight_layout(rect=(0, 0, 1, 0.9)); plt.show()

print("C3 = morph + generic, and affix is part of morph. The helper's own contribution is V3 − V2:")
for m in MODELS:
    for lang in LANGS:
        for h in HELPER[lang]:
            d = lambda part: x("V3", m, lang, f"C3_{part}_{h}") - x("V2", m, lang, f"C3_{part}_{h}")
            print(f"  {m:8s}{lang} vs {h}   V2 floor C3 {x('V2', m, lang, f'C3_total_{h}'):.3f}   "
                  f"V3 − V2: C3 {d('total'):+.3f}  morph {d('morph'):+.3f}  affix {d('affix'):+.3f}  generic {d('generic'):+.3f}")
print("C6 under V3, of the shared tokens:")
for m in MODELS:
    for lang in LANGS:
        c6 = lambda k: x("V3", m, lang, f"C6_{k}_share_of_shared")
        print(f"  {m:8s}{lang}   useful (productive suffixes) {c6('useful'):.0%} · single letters {c6('fragment'):.0%} · no letters {c6('junk'):.0%}")

**The benefit is real, but far smaller than C3 suggests.** Read the V2 row first: V2 never saw a word of
Turkish, yet most of az's tokens under it land on pieces Turkish also uses. That is the floor C3 starts from
(B1, point 3). The helper's own contribution is V3 − V2, a few points on the headline C3. Under Unigram
part of it is real morphological sharing for both languages; **under BPE the az helper adds almost nothing
alphabetic** while be's still does. So the transfer channel joint training opens is real for be,
model-dependent for az, and in every case a few percent of the token mass, not the four fifths the raw C3
reads as. `morph` is the defensible reading; C3 on its own is not.

**Is what is shared any good?** Under V3 about a quarter of the shared tokens are productive suffixes, and at
least as many are single letters. That is the first branch of the criterion, not the second: the sharing
is not pure dilution. But it buys at most a few points of affix sharing (Unigram az; next to none for be or
under BPE) for a large fertility increase, which is the trade the two recovery arms try to improve.

In [ ]:
def moves(m, lang, a, b, keys):
    h = HELPER[lang][0]
    spec = {"C1": ("C1_fertility", ".2f"), "C4": ("C4_frag4_rate", ".0%"), "C5": ("C5_alloc_share", ".1%"),
            "C3": (f"C3_total_{h}", ".2f"), "generic": (f"C3_generic_{h}", ".2f"), "morph": (f"C3_morph_{h}", ".3f"),
            "affix": (f"C3_affix_{h}", ".3f"), "single letters": ("C6_fragment_share_of_shared", ".2f")}
    return f"vs {h}  " + "  ".join(f"{k} {x(a, m, lang, spec[k][0]):{spec[k][1]}}→{x(b, m, lang, spec[k][0]):{spec[k][1]}}" for k in keys)

print("R1 · temperature sampling (V4). Gap recovered = (C1_V3 − C1_V4) / (C1_V3 − C1_V1)")
for m in MODELS:
    for lang in LANGS:
        c1 = lambda v: x(v, m, lang, "C1_fertility")
        print(f"  {m:8s}{lang}  gap recovered " + " · ".join(f"α={a} {(c1('V3') - c1('V4a' + a)) / (c1('V3') - c1('V1')):.0%}" for a in ["0.8", "0.5", "0.2"]))
        print(f"  {'':10s}V3→α=0.2 " + moves(m, lang, "V3", "V4a0.2", ["C1", "C4", "C5", "C3", "generic", "morph", "affix"]))
print("\nR2 · V5 fertility as the CSV has it (text without the character map) and as a V5 model sees it (text with it)")
for m in MODELS:
    for lang in LANGS:
        lines = [l.strip() for l in (PROC / "vocab_training" / f"{lang}.test.txt").read_text(encoding="utf-8").splitlines() if l.strip()]
        mapped = sum(map(len, ted.segment(lines, lang, f"V5_8000_{m}"))) / sum(len(l.split()) for l in lines)
        print(f"  {m:8s}{lang}  C1  V3 {x('V3', m, lang, 'C1_fertility'):.2f}   "
              f"V5 without the map {x('V5', m, lang, 'C1_fertility'):.2f}   V5 with the map {mapped:.2f}")

**R1, temperature sampling, recovers most of the cost and keeps the morphology.** At α = 0.2 az gets most of
its fertility gap back, the share of shattered words roughly halves and its slot share rises by an order of
magnitude. The headline C3 falls, **but the fall is entirely in the generic part**: `morph` against Turkish
is essentially unchanged (and rises under BPE), and only its affix subset loses a little. Temperature
sampling removes the free script overlap and keeps the part that carries meaning. This is the result to
lead with. For be the fertility recovery is similar, but its affix sharing drops by a sixth to a quarter: be pays
more for R1 than az does, plausibly because its helpers are closer and its affix sharing was real to begin
with. Report the two languages separately rather than averaging them.

**R2, script unification, is not measured correctly yet.** V5 is trained on text after the character map
(`translit` in `train_vocabs.py`: ə/x/q → e/h/k for az, ў/і/э → у/и/е for be), but `cost_analysis.py` and
`directionality.py` evaluate it on text *without* the map. V5 then meets letters its training rarely or never
saw (under BPE many become unknown pieces), so every V5 number in the two CSVs, including the V5 points in
both figures and the V5 column of `sensitivity.md`, measures that mismatch, not script unification. Measured
the way a V5 model sees its input (last lines above), V5's fertility is *below* V3's for both languages and
both models: R2 may recover part of the cost after all. Before any R2 claim, both scripts have to map the
evaluation text for V5 and `cost` / `directionality` have to be re-run. What fertility cannot show is the
price: the map is lossy (ə and e become one letter), so an `en→az` V5 model writes text that cannot be mapped
back exactly, and evaluation has to decide how that direction is scored.

In [ ]:
u = lambda v, lang, col: x(v, "unigram", lang, col)
span = lambda vals, fmt: f"{min(vals):{fmt}}–{max(vals):{fmt}}"
at_v2 = [(u("V2", l, "C1_fertility") - u("V1", l, "C1_fertility")) / (u("V3", l, "C1_fertility") - u("V1", l, "C1_fertility")) for l in LANGS]
gap = [(u("V3", l, "C1_fertility") - u("V4a0.2", l, "C1_fertility")) / (u("V3", l, "C1_fertility") - u("V1", l, "C1_fertility")) for l in LANGS]
helped = lambda m, part: x("V3", m, "az", f"C3_{part}_tr") - x("V2", m, "az", f"C3_{part}_tr")
display(Markdown(f"""**The one-paragraph version** (Unigram; every number read from the CSVs above)

> A subword vocabulary learned jointly with larger languages costs Azerbaijani and Belarusian
> ×{span([u("V3", l, "C1_fertility") / u("V1", l, "C1_fertility") for l in LANGS], ".1f")} in fertility and
> ×{span([u("V3", l, "C2_inflation_vs_baseline") for l in LANGS], ".1f")} in sequence length, and raises the share of words
> shattered into four or more pieces from {span([u("V1", l, "C4_frag4_rate") for l in LANGS], ".0%")} to
> {span([u("V3", l, "C4_frag4_rate") for l in LANGS], ".0%")}. The mechanism is slot allocation (az holds
> {u("V3", "az", "C5_alloc_share"):.1%} of an 8,000-piece joint vocabulary against {u("V1", "az", "C5_alloc_share"):.0%} of its own),
> and {span(at_v2, ".0%")} of the fertility cost is incurred by sharing with English alone. The transfer channel the
> helper languages open is real but small: they add {helped("unigram", "total"):+.2f} to the share of az tokens landing on
> pieces Turkish also uses ({helped("unigram", "morph"):+.2f} of it alphabetic under Unigram, {helped("bpe", "morph"):+.2f} under BPE),
> on top of a {u("V2", "az", "C3_total_tr"):.2f} floor that any two Latin-script translations of the same English share for free.
> Temperature sampling at α = 0.2 recovers {gap[0]:.0%} (az) / {gap[1]:.0%} (be) of the fertility gap while leaving
> morphological sharing intact. Script unification (R2) awaits a corrected measurement."""))

**What this does not establish.** Nothing here is measured at the level of translation quality. The V1–V5
segmentations are what the from-scratch transformer trains on, and BLEU / chrF is that track's result, not
ours. The NLLB track keeps NLLB's own SentencePiece vocabulary, so V1–V5 do not apply to it and the two
tracks' BLEU is not comparable. The corpus is a translation corpus (B1), so "similarity" between helpers and
targets is partly one shared translation process. The corpora are tiny (B1), the vocabulary size is 8,000
only, and the affix threshold and the transliteration table are linguistic judgments to confirm with the
group. `C6_useful_share_of_nonshared` is not usable where almost every token is shared (V5, V3 be under
BPE): its denominator is a few dozen tokens.

### B3 · Which of these numbers survive a change of setting

Every number above rests on a setting someone chose: how much helper text defines "shared", what counts as a
productive affix, which split is evaluated, how much corpus C5 samples. `sensitivity.py` re-runs
`cost_analysis.py` under varied flags; its verdict is below and the full tables are in
`results/metrics/preprocessing/sensitivity.md`. It fixes how each kind of number may be quoted:

| Kind | Examples | Quote it |
|---|---|---|
| does not use the setting at all | C3, `generic ÷ C3`, `morph` | directly |
| defined by the setting | `affix` | with its threshold (`--min-suffix-support`) |
| depends on the sampling protocol | C5 | as a range, with the protocol; never as a single value |

In [ ]:
report = (RESULTS / "sensitivity.md").read_text(encoding="utf-8")
display(Markdown(report[report.index("## Verdict"):].replace("## Verdict", "#### Verdict of `sensitivity.py`", 1)))